# ⚙️ Notebook 3 : Pré-traitement & Pipeline sans Fuite de Données
**Projet :** Prédiction de l'Attrition des Employés — Data Mining  
**Objectif :** Conception d'un pipeline d'encodage et de mise à l'échelle rigoureux avec `ColumnTransformer`, partitionnement stratifié 80/20 et garantie d'absence de fuite de données (Data Leakage).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Modules de pré-traitement importés.")

## 1. Définition des Variables et Chargement

In [ ]:
DATA_PATH = Path("../data/raw/Employee_Attrition_DataSet.csv")
df = pd.read_csv(DATA_PATH)

NUMERICAL_FEATURES = [
    "Age", "JobLevel", "YearsAtCompany", "MonthlyIncome",
    "JobSatisfaction", "WorkLifeBalance", "DistanceFromHome",
    "PerformanceRating", "TrainingHoursLastYear"
]
CATEGORICAL_FEATURES = ["Department", "OverTime", "PromotionLast5Years"]
TARGET = "Attrition"

# Préparation X et y
X = df[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET].map({"Yes": 1, "No": 0})

print(f"X : {X.shape} | y : {y.shape} (Taux classe 1: {y.mean()*100:.2f}%)")

## 2. Partitionnement Stratifié Train / Test (80/20)
La séparation intervient **avant** toute transformation afin de garantir l'étanchéité absolue du jeu de test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Ensemble d'entraînement : {X_train.shape[0]} observations")
print(f"Ensemble de test : {X_test.shape[0]} observations")
print(f"Proportion Attrition Train : {y_train.mean()*100:.2f}% | Test : {y_test.mean()*100:.2f}%")

## 3. Construction du Préprocesseur (ColumnTransformer)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERICAL_FEATURES),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES)
    ],
    remainder="drop"
)
preprocessor

## 4. Ajustement sur Train et Transformation

In [ ]:
# Ajustement UNIQUEMENT sur X_train
preprocessor.fit(X_train)

# Transformation des deux jeux
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Nombre total de variables transformées : {len(feature_names)}")
print("Liste des variables créées :")
for f in feature_names:
    print(" -", f)

## 5. Contrôle de Non-Fuite (Moyenne & Variance)

In [ ]:
# Vérification que le StandardScaler est calé sur le Train
df_num_train = pd.DataFrame(X_train_trans, columns=feature_names)
print("Moyennes sur le Train (devraient être ~ 0) :")
print(df_num_train.iloc[:, :len(NUMERICAL_FEATURES)].mean().round(4).to_dict())

print("\nÉcarts-types sur le Train (devraient être ~ 1) :")
print(df_num_train.iloc[:, :len(NUMERICAL_FEATURES)].std().round(4).to_dict())